# Purpose of the notebook

In this notebook we format the adata formatted xenium datasets to csv's that we will later use as an input for the tasks that require R (i.e. Seurat pipelines)

# Import packages

In [1]:
import numpy as np
from pathlib import Path
import os
# import sys

import pandas as pd

import anndata as ad
import scanpy as sc

import seaborn as sns
import matplotlib.pyplot as plt

import xb.plotting as xp


## Define main paths

In [2]:
!pwd

/home/flavio/uv/Xenium_benchmarking/notebooks/0_formatting


In [3]:
maindir= Path('../../data/unprocessed_adata/')
output_dir=Path('../../data/unprocessed_adata_nuclei/') 

files=os.listdir(maindir)
files

['README.md']

```Bash
wget -c https://zenodo.org/records/11120307/files/ms_brain_multisection1.h5ad?download=1 -O ms_brain_multisection1.h5ad

```

In [4]:
files=['ms_brain_multisection1.h5ad', 'human_brain.h5ad','ms_brain_multisection2.h5ad',
 'ms_brain_multisection3.h5ad','realmouse_1.h5ad', 'realmouse_2.h5ad', 'realmouse_3.h5ad',
 'realmouse_4.h5ad', 'hbreast_ilc_addon_set2.h5ad', 'hbreast_ilc_addon_set4.h5ad','hbreast_ilc_entiresample_set3.h5ad',
 'healthy_lung.h5ad', 'human_alzheimers.h5ad', 'human_gbm.h5ad','human_spinal_chord_active.h5ad',
 'human_spinal_chord_inactive.h5ad','h_breast_1.h5ad','h_breast_2.h5ad','lung_cancer.h5ad',
 'ms_brain_fullcoronal.h5ad','ms_brain_partialcoronal.h5ad']

### IDs x files

```Bash
for id in 11121221 11124988; do
  echo "== $id"
  curl -s "https://zenodo.org/records/$id" | grep -oP "/records/$id/files/\K[^\"?]+" | sort -u
done


== 11121221
h_breast_2.h5ad
h_breast_2.h5ad/content
hbreast_idc_addon_set2.h5ad
hbreast_idc_addon_set2.h5ad/content
hbreast_idc_addon_set4.h5ad
hbreast_idc_addon_set4.h5ad/content
hbreast_idc_entiresample_set3.h5ad
hbreast_idc_entiresample_set3.h5ad/content
hbreast_ilc_addon_set2.h5ad
hbreast_ilc_addon_set2.h5ad/content
hbreast_ilc_addon_set4.h5ad
hbreast_ilc_addon_set4.h5ad/content
hbreast_ilc_entiresample_set3.h5ad
hbreast_ilc_entiresample_set3.h5ad/content
ms_brain_fullcoronal.h5ad
ms_brain_fullcoronal.h5ad/content


== 11124988
h_breast_1.h5ad
h_breast_1.h5ad/content
healthy_lung.h5ad
healthy_lung.h5ad/content
human_alzheimers.h5ad
human_alzheimers.h5ad/content
human_brain.h5ad
human_brain.h5ad/content
human_gbm.h5ad
human_gbm.h5ad/content
human_spinal_chord_active.h5ad
human_spinal_chord_active.h5ad/content
human_spinal_chord_inactive.h5ad
human_spinal_chord_inactive.h5ad/content
lung_cancer.h5ad
lung_cancer.h5ad/content
```

### Dowload file from Zenodo

```Bash

for id in 11121221 11124988; \
  do echo "== $id"; \
   curl -s "https://zenodo.org/records/$id" | grep -oP "/records/$id/files/\K[^\"?]+" | sort -u; \
  done


for f in ms_brain_multisection1 ms_brain_multisection2 ms_brain_multisection3 ms_brain_partialcoronal  realmouse_1 realmouse_2 realmouse_3 realmouse_4; 
do   
   curl -fL -C - -o "${f}.h5ad"  "https://zenodo.org/records/11120307/files/${f}.h5ad?download=1"     || echo "FAILED: $f";
done

ls -lstrh
total 45G
4,0K -rw-rw-r-- 1 flavio flavio  198 Sep  1 16:06 README.md
5,4G -rw-rw-r-- 1 flavio flavio 5,4G Sep  1 19:30 ms_brain_multisection2.h5ad
5,8G -rw-rw-r-- 1 flavio flavio 5,8G Sep  1 19:31 ms_brain_multisection1.h5ad
5,5G -rw-rw-r-- 1 flavio flavio 5,5G Sep  1 20:00 ms_brain_multisection3.h5ad
927M -rw-rw-r-- 1 flavio flavio 927M Sep  1 20:12 ms_brain_partialcoronal.h5ad
6,0G -rw-rw-r-- 1 flavio flavio 6,0G Sep  1 20:23 realmouse_1.h5ad
7,2G -rw-rw-r-- 1 flavio flavio 7,2G Sep  1 20:51 realmouse_2.h5ad
7,0G -rw-rw-r-- 1 flavio flavio 7,0G Sep  1 21:04 realmouse_3.h5ad
7,5G -rw-rw-r-- 1 flavio flavio 7,5G Sep  1 23:03 realmouse_4.h5ad


REC=11121221
for f in h_breast_2 hbreast_idc_addon_set2 hbreast_idc_addon_set4 hbreast_idc_entiresample_set3 \
         hbreast_ilc_addon_set2 hbreast_ilc_addon_set4 hbreast_ilc_entiresample_set3 ms_brain_fullcoronal; do
  curl -fL -C - -o "${f}.h5ad"  "https://zenodo.org/records/${REC}/files/${f}.h5ad?download=1" || echo "FAILED: $f"
done

REC=11124988
for f in h_breast_1 healthy_lung human_alzheimers human_brain human_gbm \
         human_spinal_chord_active human_spinal_chord_inactive lung_cancer; do
  curl -fL -C - -o "${f}.h5ad" \
    "https://zenodo.org/records/${REC}/files/${f}.h5ad?download=1" || echo "FAILED: $f"
done

```

In [5]:
# did not work
# !python download_xenium_adata.py

## main loop

In [21]:
from typing import Any

root_unproc = Path("../../data/formatted_for_R/unprocessed_adata_nuclei/")

def read_h5ad(filename: Path) -> Any:
    try:
        adata = ad.read_h5ad(str(filename))
        print(f"Ok reading")
    except:
        adata = None
        print(f"Error reading")

    return adata



In [16]:
filename = output_dir / "h_breast_1.h5ad"
filename = output_dir / "h_breast_2.h5ad"
filename = output_dir / files[4]
filename = output_dir / "human_brain.h5ad"

try:
    if filename.exists():
        adata = read_h5ad(filename)
except:
    print(f"Error reading file: {filename}")

Ok reading


In [17]:
# adata.__dict__

In [18]:
output_dir

PosixPath('../../data/unprocessed_adata_nuclei')

In [19]:
for fname in files:
    filename = output_dir / fname

    if filename.exists():
        try:
            print(filename.exists(), filename)
        except:
            print(f"Error reading file: {filename}")
            continue
    else:
        print(f"Warning: file not found: {filename}")
                        

True ../../data/unprocessed_adata_nuclei/ms_brain_multisection1.h5ad
True ../../data/unprocessed_adata_nuclei/human_brain.h5ad
True ../../data/unprocessed_adata_nuclei/ms_brain_multisection2.h5ad
True ../../data/unprocessed_adata_nuclei/ms_brain_multisection3.h5ad
True ../../data/unprocessed_adata_nuclei/realmouse_1.h5ad
True ../../data/unprocessed_adata_nuclei/realmouse_2.h5ad
True ../../data/unprocessed_adata_nuclei/realmouse_3.h5ad
True ../../data/unprocessed_adata_nuclei/realmouse_4.h5ad
True ../../data/unprocessed_adata_nuclei/hbreast_ilc_addon_set2.h5ad
True ../../data/unprocessed_adata_nuclei/hbreast_ilc_addon_set4.h5ad
True ../../data/unprocessed_adata_nuclei/hbreast_ilc_entiresample_set3.h5ad
True ../../data/unprocessed_adata_nuclei/healthy_lung.h5ad
True ../../data/unprocessed_adata_nuclei/human_alzheimers.h5ad
True ../../data/unprocessed_adata_nuclei/human_gbm.h5ad
True ../../data/unprocessed_adata_nuclei/human_spinal_chord_active.h5ad
True ../../data/unprocessed_adata_nucle

In [22]:
for fname in files:
    filename = output_dir / fname

    print(f"{str(filename).split('/')[-1]:40}", end=' ')
    filename2 = ''

    if filename.exists():
        try:
            tag=fname.split('.')[0]
            root_tag = root_unproc / tag

            if not root_tag.exists():
                os.mkdir(root_tag)

            adata = None

            filename2 = root_tag / 'obs.csv'

            if not filename2.exists():
                if not adata:
                    adata = read_h5ad(filename)
                    if not adata: continue
                
                adata.obs.to_csv(filename2)

            filename2 = root_tag / 'var.csv'
            if not filename2.exists():
                if not adata:
                    adata = read_h5ad(filename)
                    if not adata: continue
                
                adata.var.to_csv(filename2)

            filename2 = root_tag / 'exp.csv'
            if not filename2.exists():
                if not adata:
                    adata = read_h5ad(filename)
                    if not adata: continue
                
                adata.to_df().to_csv(filename2)

            if not adata:
                print("")

        except:
            print(f"Error writing file: {filename2}")
            continue
    else:
        print(f"Not found")
                        

ms_brain_multisection1.h5ad              
human_brain.h5ad                         
ms_brain_multisection2.h5ad              
ms_brain_multisection3.h5ad              
realmouse_1.h5ad                         
realmouse_2.h5ad                         
realmouse_3.h5ad                         
realmouse_4.h5ad                         
hbreast_ilc_addon_set2.h5ad              Error reading
hbreast_ilc_addon_set4.h5ad              Error reading
hbreast_ilc_entiresample_set3.h5ad       Error reading
healthy_lung.h5ad                        Error reading
human_alzheimers.h5ad                    
human_gbm.h5ad                           
human_spinal_chord_active.h5ad           Error reading
human_spinal_chord_inactive.h5ad         Error reading
h_breast_1.h5ad                          Error reading
h_breast_2.h5ad                          
lung_cancer.h5ad                         Error reading
ms_brain_fullcoronal.h5ad                Error reading
ms_brain_partialcoronal.h5ad             
